# 05 - Business Insights

**Objective:** Bring together the findings from notebooks 01-04 into a
single set of business-ready conclusions and recommendations, each tied to
a specific calculated result.

**Note:** this notebook does not run new analysis -- it synthesizes results
already computed and validated in the earlier notebooks and in the SQL
layer (`sql/`). See `docs/findings.md` and `docs/recommendations.md` for the
full write-up.


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data_loading import load_orders
from data_cleaning import clean_orders
from feature_engineering import engineer_all
from kpi_calculations import compute_core_kpis

raw = load_orders()
cleaned, _ = clean_orders(raw)
tables = engineer_all(cleaned)
lines = tables['lines']
kpis = compute_core_kpis(lines)

print("HEADLINE KPIs")
print("="*50)
for k, v in kpis.items():
    print(f"{k:>26}: {v:,.4f}" if isinstance(v, float) else f"{k:>26}: {v:,}")


HEADLINE KPIs
             total_revenue: 2,326,534.3543
              total_profit: 292,296.8146
              total_orders: 5,111
               total_units: 38,654
             profit_margin: 0.1256
       average_order_value: 455.2014
   average_units_per_order: 7.5629
            customer_count: 804
      revenue_per_customer: 2,893.6994
      repeat_customer_rate: 0.9851


## Summary of findings

**1. Overall performance is healthy and growing.**
Total revenue is $2.33M with $292K profit (12.56% margin) across 5,111
orders from 804 customers. Revenue grew in 3 of the last 4 years, with 2025
and 2026 both showing 20%+ YoY growth.

**2. Furniture is a volume business with a margin problem.**
Furniture generates $754,748 in revenue (2nd of 3 categories) but only
$19,730 in profit (2.6% margin) -- the lowest of the three categories by a
wide margin. Technology and Office Supplies both run ~17% margins on
similar or lower revenue.

**3. Discount levels above ~20% are associated with negative margins.**
Margin is positive at every discount tier from 0-20%, and negative at
every tier from 20% upward, with a -0.865 correlation between discount and
line-item margin. This is a strong *association*; the analysis does not
establish that discounting *causes* the margin loss (see caveat below).

**4. Revenue and profit rank are not the same thing, at every level.**
- The single highest-revenue customer is unprofitable (-$1,981 profit
  on $25,043 revenue).
- The #3 highest-revenue product (Cisco TelePresence System EX90) has a
  -8.0% margin.
- Texas is a top-3 state by revenue ($170,188) but the single largest
  loss-making state in the dataset (-$25,729 profit).

**5. Revenue is only moderately concentrated among customers.**
The top 10% of customers by revenue generate ~31% of total revenue --
growth is broad-based, not dependent on a handful of accounts.


## Explicit limitation

**No unit-cost data exists in this dataset.** Every margin figure in this
project is `Profit / Sales` (a revenue-based margin), calculated directly
from the `Profit` column that Superstore already provides. A cost-based
margin (`Profit / Cost of Goods`) cannot be computed and is never implied.


## Where management should investigate first

Ranked by a combination of dollar impact and how actionable the lever is:

1. **Discount policy above 20%**, especially in Office Supplies (widest
   swing from +36.8% margin at 0% discount to -121.6% margin above 30%).
2. **Furniture category margin**, particularly the Tables sub-category
   (-8.5% margin, -$17,753 total loss).
3. **Texas and other loss-making high-revenue states** (Texas, Ohio,
   Pennsylvania, Illinois together account for -$70,868 in losses).
4. **Products that are simultaneously top-sellers and loss-makers**
   (see notebook 04's "high-sales, low-margin" table).
